<a href="https://colab.research.google.com/github/RossIsland/MINLP-Surrogate-Modelling/blob/main/compressor_2_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install PyTorch, Torchvision, and Pyomo (removed the non-existent scalar_formatter)
!pip install -q torch torchvision pyomo

# 2. Install amplpy AND ampltools
!pip install -q amplpy ampltools

# 3. Install the CBC solver module directly via amplpy
!python -m amplpy.modules install cbc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.8 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_cbc
Looking in indexes: https://pypi.ampl.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 46.8 MB/s eta 0:00:00
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_cbc.


In [ ]:
# ==========================================
# CELL 2: COMPRESSOR DATA GENERATION & PREPROCESSING
# ==========================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

def simulate_compressor_physics(P_in, T_in, Q_in, s, n):
    # Performance curve models (simplified polynomial representations matching performance envelopes)
    # Outputs: P_out, T_out, Q_out, W, Q_min, Q_max

    # Calculate compression ratio scaling with rotational speed and flow rate per running unit
    q_per_comp = Q_in / max(float(n), 1.0)
    speed_ratio = s / 4800.0

    # Simple explicit head estimation and pressure drop
    head = 50.0 * (speed_ratio**2) - 0.01 * q_per_comp
    head = max(head, 10.0)

    P_out = P_in * (1.0 + (head / 150.0))
    T_out = T_in * ((P_out / P_in)**0.24)
    Q_out = Q_in - 0.005 * Q_in  # Account for minor standard fuel consumption loops if gas-driven
    W = 1.2 * head * Q_in        # Shaft power approximation

    # Boundary tracking for performance envelopes
    Q_min = 50.0 * speed_ratio
    Q_max = 2000.0 * speed_ratio

    P_out = np.clip(P_out, 4.0, 12.0)
    T_out = np.clip(T_out, 273.15, 360.0)
    return P_out, T_out, Q_out, W, Q_min, Q_max

np.random.seed(42)
num_samples = 10000

# Random uniform sampling across characteristic operational envelopes
P_in_samples = np.random.uniform(4.0, 8.0, num_samples)
T_in_samples = np.random.uniform(280.0, 310.0, num_samples)
Q_in_samples = np.random.uniform(300.0, 1500.0, num_samples)
s_samples = np.random.uniform(2880.0, 5040.0, num_samples)
n_samples = np.random.choice([1, 2, 3, 4], size=num_samples)

P_out_samples, T_out_samples, Q_out_samples = [], [], []
W_samples, Q_min_samples, Q_max_samples = [], [], []

for i in range(num_samples):
    po, to, qo, w, qmin, qmax = simulate_compressor_physics(
        P_in_samples[i], T_in_samples[i], Q_in_samples[i], s_samples[i], n_samples[i]
    )
    P_out_samples.append(po)
    T_out_samples.append(to)
    Q_out_samples.append(qo)
    W_samples.append(w)
    Q_min_samples.append(qmin)
    Q_max_samples.append(qmax)

df = pd.DataFrame({
    'P_in': P_in_samples, 'T_in': T_in_samples, 'Q_in': Q_in_samples, 's': s_samples, 'n': n_samples,
    'P_out': P_out_samples, 'T_out': T_out_samples, 'Q_out': Q_out_samples,
    'W': W_samples, 'Q_min': Q_min_samples, 'Q_max': Q_max_samples
})

X = df[['P_in', 'T_in', 'Q_in', 's', 'n']].values
Y = df[['P_out', 'T_out', 'Q_out', 'W', 'Q_min', 'Q_max']].values

X_offset, X_factor = X.mean(axis=0), X.std(axis=0)
Y_offset, Y_factor = Y.mean(axis=0), Y.std(axis=0)

X_scaled = (X - X_offset) / X_factor
Y_scaled = (Y - Y_offset) / Y_factor

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y_scaled, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

print("Compressor data generation complete! Dataset shape:", df.shape)

Compressor data generation complete! Dataset shape: (10000, 11)


In [ ]:
# ==========================================
# CELL 3: TRAINING THE ReLU NEURAL NETWORK
# ==========================================
class CompressorANN(nn.Module):
    def __init__(self):
        super(CompressorANN, self).__init__()
        self.fc1 = nn.Linear(5, 20)  # 5 inputs, 20 hidden neurons matching paper configuration
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 6)  # 6 outputs

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = CompressorANN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training Compressor Station ANN...")
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, Y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_loss = criterion(test_preds, Y_test_t)
    print(f"Final Test MSE Loss: {test_loss.item():.5f}")

W1 = model.fc1.weight.detach().numpy()   # (20, 5)
b1 = model.fc1.bias.detach().numpy()     # (20,)
W2 = model.fc2.weight.detach().numpy()   # (6, 20)
b2 = model.fc2.bias.detach().numpy()     # (6,)

Training Compressor Station ANN...
Final Test MSE Loss: 0.00852


In [ ]:
# ==========================================
# CELL 4: PYOMO MATHEMATICAL OPTIMIZATION
# ==========================================
from pyomo.environ import *
from amplpy import modules  # Imported to dynamically trace local binaries

opt_model = ConcreteModel()

# Continuous optimization variables for the compressor boundary nodes
opt_model.P_in = Var(bounds=(4.0, 8.0), initialize=6.0)
opt_model.T_in = Var(bounds=(280.0, 310.0), initialize=295.0)
opt_model.Q_in = Var(bounds=(300.0, 1500.0), initialize=800.0)
opt_model.s = Var(bounds=(2880.0, 5040.0), initialize=4000.0)
opt_model.n = Var(bounds=(1.0, 4.0), initialize=2.0)  # Continuous boundary representation

opt_model.P_out = Var(bounds=(4.0, 12.0))
opt_model.T_out = Var(bounds=(273.15, 360.0))
opt_model.Q_out = Var(bounds=(0.0, 2000.0))
opt_model.W = Var(bounds=(0.0, 100000.0))
opt_model.Q_min = Var(bounds=(0.0, 500.0))
opt_model.Q_max = Var(bounds=(0.0, 3000.0))

opt_model.neurons = RangeSet(0, 19)  # 20 hidden layer neurons
opt_model.u = Var(opt_model.neurons, bounds=(0.0, 50.0))
opt_model.delta = Var(opt_model.neurons, within=Binary)

M_ub = 100.0
M_lb = -100.0

def get_scaled_input(m):
    return [
        (m.P_in - X_offset[0]) / X_factor[0],
        (m.T_in - X_offset[1]) / X_factor[1],
        (m.Q_in - X_offset[2]) / X_factor[2],
        (m.s - X_offset[3]) / X_factor[3],
        (m.n - X_offset[4]) / X_factor[4]
    ]

def relu_bigm_rules(m, k):
    inputs = get_scaled_input(m)
    u_hat = sum(W1[k, i] * inputs[i] for i in range(5)) + b1[k]

    yield m.u[k] >= u_hat
    yield m.u[k] <= u_hat - M_lb * (1 - m.delta[k])
    yield m.u[k] <= M_ub * m.delta[k]
    yield u_hat >= M_lb * (1 - m.delta[k])

opt_model.relu_constraints = ConstraintList()
for k in opt_model.neurons:
    for c in relu_bigm_rules(opt_model, k):
        opt_model.relu_constraints.add(c)

# Output variable mapping rules (Reversing network standardizations)
def output_rule(m, out_idx, var_target):
    y_scaled = sum(W2[out_idx, k] * m.u[k] for k in m.neurons) + b2[out_idx]
    return var_target == (y_scaled * Y_factor[out_idx]) + Y_offset[out_idx]

opt_model.out_p_con = Constraint(rule=lambda m: output_rule(m, 0, m.P_out))
opt_model.out_t_con = Constraint(rule=lambda m: output_rule(m, 1, m.T_out))
opt_model.out_q_con = Constraint(rule=lambda m: output_rule(m, 2, m.Q_out))
opt_model.out_w_con = Constraint(rule=lambda m: output_rule(m, 3, m.W))
opt_model.out_qmin_con = Constraint(rule=lambda m: output_rule(m, 4, m.Q_min))
opt_model.out_qmax_con = Constraint(rule=lambda m: output_rule(m, 5, m.Q_max))

# Set fixed target operating point context
opt_model.P_in.fix(6.2)
opt_model.T_in.fix(290.0)
opt_model.Q_in.fix(900.0)
opt_model.s.fix(4500.0)
opt_model.n.fix(2.0)

opt_model.obj = Objective(expr=opt_model.W, sense=minimize)

# ========================================================
# ROBUST ENVIRONMENT RESOLUTION FOR PIPELINE OPTIMIZATION
# ========================================================

# 1. Pull the absolute file system path for the CBC executable
executable_path = modules.find("cbc")

# 2. Use 'cbcnl' ASL binary driver factory wrapper to circumvent path anomalies
solver = SolverFactory("cbcnl", executable=executable_path, solve_io="nl")

# 3. Solve the optimization model and let Pyomo apply outputs automatically
results = solver.solve(opt_model)

# ========================================================
# DISPLAY OPTIMIZATION OUTPUT
# ========================================================
print("\n--- Optimization Execution Successfully Finished ---")
print(f"Minimized Power Work (W): {value(opt_model.W):.2f}")
print(f"Predicted Outlet Pressure: {value(opt_model.P_out):.4f} MPa")
print(f"Predicted Outlet Temperature: {value(opt_model.T_out):.2f} K")


--- Optimization Execution Successfully Finished ---
Minimized Power Work (W): 42473.54
Predicted Outlet Pressure: 7.9218 MPa
Predicted Outlet Temperature: 305.84 K


In [ ]:
# ==============================================================================
# CELL 5: EXTRACT LOCALIZED LINEAR CONSTRAINTS ONLY (RIGOROUS COMPRESSOR FIX)
# ==============================================================================
import numpy as np

# Capture frozen operating configuration bounds
active_neurons = [k for k in opt_model.neurons if value(opt_model.delta[k]) > 0.5]

print(f"Active hidden layer neurons evaluated at this localized point: {active_neurons}\n")

# Setup parameter transformation lists
output_names = ["P_out", "T_out", "Q_out", "W", "Q_min", "Q_max"]

print("="*80)
print("UNIT-CORRECTED EXTRACTED COMPRESSOR PIECEWISE LINEAR EQUATIONS")
print("="*80)
print("Variables context mapping:")
print("  x[0]=P_in, x[1]=T_in, x[2]=Q_in (10^4 m3/d), x[3]=s, x[4]=n\n")

# Volumetric flow conversion factor: 10^4 m3/d to standard operational m3/s
flow_conversion = 10000.0 / 86400.0

# Loop over each of the 6 physical target equations
for idx, name in enumerate(output_names):
    # Compressor networks expect exactly 5 inputs: P_in, T_in, Q_in, s, n
    m_scaled = np.zeros(5)
    c_scaled = b2[idx]

    for k in active_neurons:
        m_scaled += W2[idx, k] * W1[k, :]
        c_scaled += W2[idx, k] * b1[k]

    # Reconvert from standardized scaling plane back to real-world physics metrics
    m_phys = (m_scaled / X_factor) * Y_factor[idx]
    c_phys = Y_offset[idx] + Y_factor[idx] * (c_scaled - sum((m_scaled * X_offset) / X_factor))

    # --- THE RIGOROUS PHYSICAL UNIT TRANSLATION ---
    # 1. Correct the sensitivity of the input variable gradient matching index x[2] (Q_in)
    m_phys[2] = m_phys[2] * flow_conversion

    # 2. If the current output expression targets a flow capacity (Q_out, Q_min, Q_max)
    # we must isolate and scale down all input slopes and the intercept constant
    if name in ["Q_out", "Q_min", "Q_max"]:
        m_phys = m_phys / flow_conversion
        c_phys = c_phys / flow_conversion

    # Print clean algebraic output string matching your exact AMPL format requirements
    print(f"{name} = ({m_phys[0]:.6e} * P_in) + ")
    print(f"          ({m_phys[1]:.6e} * T_in) + ")
    print(f"          ({m_phys[2]:.6e} * Q_in) + ")
    print(f"          ({m_phys[3]:.6e} * s) + ")
    print(f"          ({m_phys[4]:.6e} * n) + ({c_phys:.6f});\n")
print("="*80)

Active hidden layer neurons evaluated at this localized point: [0, 1, 2, 4, 8, 9, 13, 14, 18, 19]

UNIT-CORRECTED EXTRACTED COMPRESSOR PIECEWISE LINEAR EQUATIONS
Variables context mapping:
  x[0]=P_in, x[1]=T_in, x[2]=Q_in (10^4 m3/d), x[3]=s, x[4]=n

P_out = (1.134228e+00 * P_in) + 
          (8.691296e-03 * T_in) + 
          (-5.074929e-05 * Q_in) + 
          (8.641332e-04 * s) + 
          (1.620426e-01 * n) + (-5.448995);

T_out = (8.102546e-01 * P_in) + 
          (9.516915e-01 * T_in) + 
          (-4.934076e-04 * Q_in) + 
          (6.172136e-03 * s) + 
          (3.088561e-01 * n) + (0.266773);

Q_out = (-8.412206e+01 * P_in) + 
          (-1.644402e+01 * T_in) + 
          (7.843703e-01 * Q_in) + 
          (-7.549419e-01 * s) + 
          (-2.824816e+02 * n) + (11181.502490);

W = (-6.357542e+02 * P_in) + 
          (9.603108e+01 * T_in) + 
          (3.087110e+00 * Q_in) + 
          (2.040797e+01 * s) + 
          (2.241769e+03 * n) + (-101758.576837);

Q_min = (4.801316e